In [0]:
%run ../config/config

In [0]:
!pip install jinja2==3.1.6

In [0]:
import json
import pandas as pd
import numpy as np
import os
import sys
import time

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import NumericType
from datetime import datetime

sys.path.insert(0, str(PROJECT_PATH))

from utils.data_quality import DataQualityAnalyzer, quality
from utils.reporting import HTMLReportGenerator, ChartGenerator, TableBuilder, HTMLFormatter
from utils.logger import MLOpsLogger, log_dataframe_info
from utils.evolutive_batch import (
    compute_evolutive_batch,
    build_evolutive_pivots,
    compute_historical_averages,
    compute_evolutive_semaphores,
    compute_evolutive_summary_kpis
)

print("Imports completados exitosamente")

In [0]:
logger = MLOpsLogger(
    name='04_quality_inputs',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={'mes_vta': MES_VTA, 'environment': ENV, 'notebook': 'engine_quality'}
)

In [0]:
# =============================================================================
# CONFIGURACIÓN CENTRALIZADA
# =============================================================================
try:
    logger.info("=" * 60)
    logger.info("CARGANDO CONFIGURACIÓN CENTRALIZADA")
    logger.info("=" * 60)

    from utils.config_loader import ConfigLoader
    config_loader = ConfigLoader(QUALITY_CONFIG)

    all_configs = config_loader.get_all_input_quality_config()

    analysis_settings = all_configs['analysis']
    column_settings = all_configs['columns']
    chart_settings = all_configs['charts']
    report_settings = all_configs['reports']
    semaforo_thresholds = all_configs['quality_thresholds']
    evolutive_semaforo_thresholds = all_configs['evolutive_thresholds']

    meses_historicos_calidad = analysis_settings.get('meses_historicos_calidad', MESES_HISTORICOS_CALIDAD)
    rows_per_page_config = analysis_settings.get('rows_per_page', 25)
    campo_primario = column_settings.get('campo_primario', 'codclaveunicocli')
    columnas_control = column_settings.get('columnas_control', ['codclaveunicocli', 'codclavepartycli', 'codmes', 'fecproceso', 'fecdata'])
    chart_figsize = tuple(chart_settings.get('figsize', [10, 6]))
    chart_bins = chart_settings.get('bins', 30)
    chart_dpi = chart_settings.get('dpi', 100)
    titulo_quality_report = report_settings.get('titulo_quality', 'Reporte de Calidad de Datos Inputs')

    logger.info("✓ CONFIGURACIÓN CENTRALIZADA CARGADA")

except Exception as e:
    logger.log_exception(operation='load_centralized_config', exception=e, should_raise=True)

In [0]:
# =============================================================================
# ★ CARGA DESDE DELTA STAGING (plan depth = 1, sin RecursionError)
# =============================================================================
try:
    logger.log_stage_start('engine_quality', MES_VTA=MES_VTA, environment=ENV)
    start_time = time.time()

    spark = SparkSession.builder.getOrCreate()

    # ★ Path del staging creado por 01_extract_and_persist
    persist_path = f"{ADLS_LOCATION_TABLE}/quality_engine/{ENV}/{TABLE_MASTER}"

    logger.info(f"Leyendo desde Delta staging: {persist_path}")
    load_start = time.time()

    # ★ Lectura directa desde Delta particionado - plan limpio
    df_staging = spark.read.format("delta").load(persist_path)

    # Filtrar mes actual
    df_actual = df_staging.filter(F.col("codmes") == MES_VTA)
    actual_count = df_actual.count()

    if actual_count == 0:
        raise ValueError(f"No hay registros para el mes {MES_VTA} en staging")

    # Filtrar histórico (todo lo que NO es mes actual)
    df_historico = df_staging.filter(F.col("codmes") < MES_VTA)
    historico_count = df_historico.count()

    meses_historicos = sorted([r.codmes for r in df_historico.select('codmes').distinct().collect()])
    total_records = actual_count

    load_duration = time.time() - load_start
    logger.info(f"✓ Actual: {actual_count:,} | Histórico: {historico_count:,} ({len(meses_historicos)} meses) | {load_duration:.1f}s")
    logger.info(f"  ★ Plan depth = 1 (lectura directa desde Delta staging)")

except Exception as e:
    logger.log_exception(operation='load_from_staging', exception=e, should_raise=True)

In [0]:
# =============================================================================
# CONFIGURACIÓN DE VARIABLES
# =============================================================================
try:
    todas_columnas = df_actual.columns
    columnas_numericas = [f.name for f in df_actual.schema.fields if isinstance(f.dataType, NumericType)]
    variables_existentes = [c for c in todas_columnas if c not in columnas_control and c in columnas_numericas]

    logger.info(f"Variables a analizar: {len(variables_existentes)} numéricas")

    if len(variables_existentes) == 0:
        raise ValueError("No hay variables numéricas para analizar")

except Exception as e:
    logger.log_exception(operation='configure_variables', exception=e, should_raise=True)

In [0]:
# =============================================================================
# VALIDACIONES PRE-ANÁLISIS
# =============================================================================
try:
    logger.info("VALIDACIONES PRE-ANÁLISIS")

    pk_stats = df_actual.agg(
        F.sum(F.when(F.col(campo_primario).isNull(), 1).otherwise(0)).alias("null_keys"),
        F.count(campo_primario).alias("total"),
        F.approx_count_distinct(campo_primario).alias("approx_unique")
    ).collect()[0]

    if pk_stats["null_keys"] > 0:
        raise ValueError(f"Llave primaria '{campo_primario}' tiene {pk_stats['null_keys']} valores nulos")

    duplicates = pk_stats["total"] - pk_stats["approx_unique"]
    if duplicates > 0:
        logger.warning(f"~{duplicates} registros duplicados detectados (approx)")

    logger.info("✓ Validaciones completadas")

except Exception as e:
    logger.log_exception(operation='pre_analysis_validation', exception=e, should_raise=False)

In [0]:
# =============================================================================
# ANÁLISIS DE CALIDAD - DATOS ACTUALES
# =============================================================================
try:
    logger.info("ANÁLISIS DE CALIDAD - DATOS ACTUALES")
    quality_start = time.time()

    # Definir metrics_config por defecto si no existe (retrocompatibilidad)
    if 'METRICS_CONFIG' not in globals():
        METRICS_CONFIG = None

    dq_actual = quality(df_actual)
    current_metrics = dq_actual.run(
        primary_key="CODCLAVEUNICOCLI",
        columns=variables_existentes,
        semaforo_thresholds=semaforo_thresholds,
        metrics_config=METRICS_CONFIG
    )

    quality_duration = time.time() - quality_start
    logger.info(f"✓ Calidad completada en {quality_duration:.2f}s")

    # Log alertas
    alerts_found = []
    var_metrics = current_metrics['variable_metrics']
    for idx, row in var_metrics.iterrows():
        for col in var_metrics.columns:
            if col.startswith('color_semaforo_'):
                if row[col] in ['amarillo', 'rojo']:
                    alerts_found.append({'variable': row['Variable'], 'metric': col.replace('color_semaforo_', ''), 'color': row[col]})

    if alerts_found:
        logger.warning(f"⚠️ {len(alerts_found)} alertas de calidad")
    else:
        logger.info("✓ Sin alertas de calidad")

except Exception as e:
    logger.log_exception(operation='quality_analysis_current', exception=e, should_raise=True)

In [0]:
# =============================================================================
# ANÁLISIS EVOLUTIVO + SEMÁFORO vs PROMEDIO HISTÓRICO
# =============================================================================
try:
    logger.info("=" * 60)
    logger.info("ANÁLISIS EVOLUTIVO (BATCH + SEMÁFORO vs PROMEDIO HISTÓRICO)")
    evolutive_start = time.time()

    months = sorted(meses_historicos)
    logger.info(f"  Meses históricos: {len(months)}")
    if months:
        logger.info(f"  Rango: {min(months)} - {max(months)}")

    evolutive_metrics = {}
    evolutive_semaphore_results = None
    evolutive_kpis = None

    if months and historico_count > 0:
        logger.info(f"  [1/4] Batch aggregation: {len(variables_existentes)} vars x {len(months)} meses")
        full_metrics_df = compute_evolutive_batch(df_historico, variables_existentes)

        logger.info("  [2/4] Construyendo tablas evolutivas...")
        evolutive_metrics = build_evolutive_pivots(full_metrics_df)

        logger.info(f"  [3/4] Calculando promedios de {len(months)} meses históricos...")
        historical_averages = compute_historical_averages(full_metrics_df)

        logger.info(f"  [4/4] Comparando mes {MES_VTA} vs promedio histórico...")
        current_var_df = current_metrics['variable_metrics']

        evolutive_semaphore_results = compute_evolutive_semaphores(
            current_variable_metrics=current_var_df,
            historical_averages=historical_averages,
            evolutive_thresholds=evolutive_semaforo_thresholds,
            mes_vta=str(MES_VTA)
        )

        evolutive_kpis = compute_evolutive_summary_kpis(evolutive_semaphore_results)

        if evolutive_kpis:
            overall = evolutive_kpis.get('overall', {})
            logger.info(f"  🟢 Verde: {overall.get('verde', 0)} | 🟡 Amarillo: {overall.get('amarillo', 0)} | 🔴 Rojo: {overall.get('rojo', 0)}")
    else:
        logger.warning("Sin datos históricos para análisis evolutivo")

    evolutive_duration = time.time() - evolutive_start
    logger.info(f"✓ Evolutivo completado en {evolutive_duration:.2f}s")

except Exception as e:
    logger.log_exception(operation='evolutive', exception=e, should_raise=True)

In [0]:
# =============================================================================
# ★ DISTRIBUCIONES - SQL PURO sobre Delta staging
# =============================================================================
try:
    logger.info("=" * 60)
    logger.info("SERIALIZACIÓN DE DISTRIBUCIONES")
    distribution_start = time.time()

    if 'utils.metadata_serializer' in sys.modules:
        del sys.modules['utils.metadata_serializer']
    from utils.metadata_serializer import MetadataSerializer

    metadata_dir = os.path.join(TEMP_REPORTS_INPUT_PATH, "metadata")
    os.makedirs(metadata_dir, exist_ok=True)

    serializer = MetadataSerializer(metadata_dir)

    # ★ Registrar df_actual como temp view para SQL puro
    # df_actual ya es plan depth=1 (viene de Delta staging)
    df_actual.createOrReplaceTempView("__tmp_dist_actual")

    distribution_data_path = serializer.serialize_distribution_data(
        spark=spark,
        table_name="__tmp_dist_actual",
        filter_sql=None,
        variables=variables_existentes,
        bins=chart_bins,
        filename="distribution_data.json"
    )

    try:
        spark.catalog.dropTempView("__tmp_dist_actual")
    except Exception:
        pass

    distribution_duration = time.time() - distribution_start
    logger.info(f"✓ Distribuciones en {distribution_duration:.2f}s")

except Exception as e:
    logger.log_exception(operation='distributions', exception=e, should_raise=True)

In [0]:
# =============================================================================
# GUARDAR METADATOS DE CALIDAD
# =============================================================================
try:
    logger.info("GUARDANDO METADATOS DE CALIDAD")
    save_start = time.time()

    quality_metadata_path = serializer.serialize_quality_metrics(
        current_metrics, filename="quality_metrics.json"
    )
    logger.info(f" ✓ Quality metrics: {quality_metadata_path}")

    evolutive_metadata_path = serializer.serialize_evolutive_metrics(
        evolutive_metrics, filename="evolutive_metrics.json"
    )
    logger.info(f" ✓ Evolutive metrics: {evolutive_metadata_path}")

    if evolutive_semaphore_results is not None and not evolutive_semaphore_results.empty:
        evol_sem_path = serializer.serialize_quality_metrics(
            {'evolutive_semaphores': evolutive_semaphore_results,
             'evolutive_kpis': evolutive_kpis},
            filename="evolutive_semaphores.json"
        )
        logger.info(f" ✓ Evolutive semaphores: {evol_sem_path}")

    quality_bundle_path = serializer.save_metadata_bundle(
        mes_vta=MES_VTA, environment=ENV, table_name=TABLE_MASTER,
        total_records=actual_count, variables_analyzed=variables_existentes,
        current_metrics=current_metrics, evolutive_metrics=evolutive_metrics,
        drift_results=None, variable_comparison=None,
        semaphore_kpis=None, semaphore_config=None,
        distribution_data_path=distribution_data_path,
        additional_metadata={
            'meses_historicos': len(months) if 'months' in dir() else 0,
            'meses_historicos_list': months if 'months' in dir() else [],
            'alerts_found': len(alerts_found),
            'chart_bins': chart_bins,
            'rows_per_page': rows_per_page_config,
            'evolutive_kpis': evolutive_kpis if evolutive_kpis else {}
        },
        filename="quality_metadata.json"
    )
    logger.info(f" ✓ Quality bundle: {quality_bundle_path}")
    save_duration = time.time() - save_start

    # =============================================================================
    # RESUMEN FINAL
    # =============================================================================
    total_duration = time.time() - start_time

    logger.info("=" * 60)
    logger.info("ENGINE QUALITY COMPLETADO")
    logger.info("=" * 60)
    logger.info(f"Duration total: {total_duration:.2f}s")
    logger.info(f"  Carga: {load_duration:.1f}s")
    logger.info(f"  Quality: {quality_duration:.1f}s")
    logger.info(f"  Evolutivo: {evolutive_duration:.1f}s")
    logger.info(f"  Distribuciones: {distribution_duration:.1f}s")
    logger.info(f"  Metadatos: {save_duration:.1f}s")
    logger.info(f"Variables: {len(variables_existentes)}")

    logger.log_stage_end('engine_quality', duration=total_duration,
                        records_processed=actual_count,
                        variables_analyzed=len(variables_existentes))

except Exception as e:
    logger.log_exception(operation='save_quality_metadata', exception=e, should_raise=True)